# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides an example workflow for loading and exploring the [FAIR²](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library.

### Dataset Source
The dataset is provided as a Croissant schema via the following URL:
`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets (tables), fields (columns), and their IDs.

We will list all record sets, their `@id`, and summarize their columns/fields.

In [ ]:
# List all RecordSets and their fields by @id
record_set_ids = [rs['@id'] for rs in metadata.to_json().get('recordSet', [])]
if not record_set_ids:
    # fallback: try 'recordSets'
    record_set_ids = [rs['@id'] for rs in metadata.to_json().get('recordSets', [])]

if not record_set_ids:
    print('No record sets found in metadata. The dataset may not define explicit record sets.')
else:
    for rs in metadata.to_json()['recordSet']:
        print(f"RecordSet name: {rs.get('name', 'N/A')}, @id: {rs['@id']}")
        print(f"\tDescription: {rs.get('description','')}")
        print(f"\tFields (Columns):")
        for field in rs.get('field', []):
            fid = field['@id'] if isinstance(field, dict) and '@id' in field else field
            print(f"\t  - {fid}")

### Inspect example records for a record set

Replace the value of `example_record_set_id` with the `@id` of one of the available record sets from above.

In [ ]:
# Example: Select the first available record set for demonstration
if record_set_ids:
    example_record_set_id = record_set_ids[0]
    print(f'Example records for record set @id: {example_record_set_id}')
    for i, record in enumerate(dataset.records(record_set=example_record_set_id)):
        print(record)
        if i >= 2:
            break
else:
    print("No record sets available to iterate.")

## 3. Data Extraction
Load data from the record set(s) into DataFrames for analysis. Use the record set and field `@id`s.

In [ ]:
# Extract data from each record set
dataframes = {}

for rs_id in record_set_ids:
    print(f"Loading records for record set: {rs_id}")
    records = list(dataset.records(record_set=rs_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Columns in record set {rs_id} :\n", df.columns.tolist())
        display(df.head())
    else:
        print(f"No records found for record set {rs_id}")

# For main analysis, pick the first table loaded with rows.
main_rs_id = None
for rs_id in record_set_ids:
    if rs_id in dataframes and len(dataframes[rs_id]) > 0:
        main_rs_id = rs_id
        break

if main_rs_id:
    print(f"Proceeding with record set: {main_rs_id}")
else:
    print("No dataframes available for analysis.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records, normalizing numeric fields, outlier removal, or grouping data by attributes.
You can customize the numeric field(s) and grouping as relevant.

In [ ]:
# --- Example EDA: Pick a numeric field automatically if available ---
if main_rs_id:
    df = dataframes[main_rs_id].copy()

    # Infer numeric fields
    numeric_candidates = df.select_dtypes(include=[np.number]).columns.tolist()
    if not numeric_candidates:
        # Try to coerce columns
        for col in df.columns:
            try:
                df[col] = pd.to_numeric(df[col])
            except Exception:
                pass
        numeric_candidates = df.select_dtypes(include=[np.number]).columns.tolist()

    print('Numeric fields detected:', numeric_candidates)

    if numeric_candidates:
        numeric_field = numeric_candidates[0]
        print(f"Using field '{numeric_field}' for numeric analysis.")

        threshold = df[numeric_field].mean() if df[numeric_field].notnull().sum()>0 else 0  # use average as a threshold
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalize the numeric field
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try to group by a plausible categorical field
        group_candidates = [c for c in df.columns if c != numeric_field and df[c].dtype == object]
        group_field = group_candidates[0] if group_candidates else None
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
            print(f"Grouped mean {numeric_field} by '{group_field}':")
            display(grouped_df.head())
        else:
            print("No categorical field available for grouping.")
    else:
        print("No numeric fields detected. Exploratory analysis is limited.")
else:
    print("No dataframe available for EDA.")

## 5. Visualization
Visualize the distribution of a numeric field or relationships between fields using matplotlib or seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_rs_id and numeric_candidates:
    # Histogram of the main numeric field
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    # Boxplot by group
    if group_field:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=30, ha='right')
        plt.show()
else:
    print("No numeric field or dataframe available for visualization.")

## 6. Conclusion
In this notebook, we demonstrated how to load, inspect, and perform basic exploration on the `Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells` FAIR² dataset using the `mlcroissant` library. The Croissant specification enables you to access record sets, fields, and their data using robust identifiers (`@id`).

**Summary:**
- Loaded metadata and records via the Croissant schema URL.
- Listed available record sets and explored their structure using their `@id`.
- Loaded tables into DataFrames for inspection.
- Performed simple filtering, normalization, and grouped aggregation on numeric fields.
- Visualized data distributions.

Continue exploring the dataset by analyzing other record sets or customizing the EDA to your research interests.